# CineScope: Core Analytics

**Local Spark only.** Reads corrected silver `movies_awards_enriched` data and writes report-ready charts under `outputs/charts/generated/`.

Insights:
1. Genre and decade rating patterns
2. Director prior track record versus film rating
3. Runtime profile with sample-size controls
4. High-rating, low-vote niche candidates
5. Hit-rate lift across a stable pre-release experience signal

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
load_dotenv(REPO / ".env")

from pyspark.sql import functions as F

from cinescope.analytics import (
    director_prior_correlation,
    director_prior_vs_rating,
    genre_decade_stats,
    pre_release_signal_lift,
    rating_anomalies,
    runtime_bucket_stats,
)
from cinescope.ml.features import (
    validate_feature_artifact_metadata,
    with_hit_label,
)
from cinescope.paths import get_paths
from cinescope.spark_session import build_spark_session

paths = get_paths(create_dirs=True, validate_mount=True)
spark = build_spark_session(app_name="cinescope-core-analytics", paths=paths)
spark.sparkContext.setLogLevel("WARN")

charts_dir = REPO / "outputs" / "charts" / "generated"
metrics_dir = REPO / "outputs" / "metrics"
charts_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)


def save_and_show(fig, path: Path):
    """Write PNG and display inline in the notebook."""
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", path)


cast_metrics_path = metrics_dir / "cast_crew_metrics.json"
if not cast_metrics_path.exists():
    raise RuntimeError("Run the corrected cast/crew and Oscar jobs before analytics")
validate_feature_artifact_metadata(json.loads(cast_metrics_path.read_text()))

movies = spark.read.parquet(str(paths.movies_awards_enriched_dir))
n = movies.count()
print("rows:", n)
print("path:", paths.movies_awards_enriched_dir)
movies.printSchema()

In [ ]:
oscar_metrics_path = metrics_dir / "oscar_metrics.json"
if not oscar_metrics_path.exists():
    raise RuntimeError("Run the corrected Oscar job before analytics")
validate_feature_artifact_metadata(json.loads(oscar_metrics_path.read_text()))

## 1. Genre × decade

In [ ]:
gdec = genre_decade_stats(movies)
gdec_pd = gdec.filter(F.col("film_count") >= 50).toPandas()
top_genres = (
    gdec_pd.groupby("genre")["film_count"].sum().sort_values(ascending=False).head(8).index.tolist()
)
plot_df = gdec_pd[gdec_pd["genre"].isin(top_genres)]

fig, ax = plt.subplots(figsize=(10, 5))
for genre, part in plot_df.groupby("genre"):
    part = part.sort_values("decade")
    ax.plot(part["decade"], part["median_rating"], marker="o", label=genre)
ax.set_xlabel("Decade")
ax.set_ylabel("Median IMDb rating")
ax.set_title("Median rating by genre and decade (n≥50 per cell)")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
fig.tight_layout()
p1 = charts_dir / "genre_decade_median_rating.png"
save_and_show(fig, p1)
gdec_pd.head(12)

## 2. Director prior vs rating

In [ ]:
corr = director_prior_correlation(movies)
dir_pd = director_prior_vs_rating(movies).sample(False, 0.05, seed=42).toPandas()
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(dir_pd["director_prior_rating_mean"], dir_pd["average_rating"], s=8, alpha=0.25)
ax.set_xlabel("Director prior mean rating (leakage-safe)")
ax.set_ylabel("Film average rating")
ax.set_title(f"Director track record vs film rating (corr={corr:.3f}, 5% sample)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
p2 = charts_dir / "director_prior_vs_rating.png"
save_and_show(fig, p2)
print("corr", corr)

## 3. Runtime profile

This is a descriptive association, not evidence that changing runtime causes a rating change. Buckets with fewer than 50 films are excluded.

In [ ]:
rt = runtime_bucket_stats(movies, min_films=50).toPandas()
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(
    rt["runtime_bucket"],
    rt["film_count"],
    width=12,
    alpha=0.35,
    label="film count",
)
ax2 = ax1.twinx()
ax2.plot(
    rt["runtime_bucket"],
    rt["median_rating"],
    color="C3",
    marker="o",
    label="median rating",
)
ax1.set_xlabel("Runtime bucket (minutes)")
ax1.set_ylabel("Films")
ax2.set_ylabel("Median rating")
ax1.set_title("Runtime profile: volume and median rating (n >= 50 per bucket)")
fig.tight_layout()
p3 = charts_dir / "runtime_profile.png"
save_and_show(fig, p3)
rt.sort_values("film_count", ascending=False).head(5)

## 4. Rating anomalies (high rating, low votes)

In [ ]:
anom = rating_anomalies(movies, high_rating=8.0, vote_percentile=0.10)
anom_n = anom.count()
anom_pd = anom.limit(5000).toPandas()
sample_all = movies.select("average_rating", "num_votes").sample(False, 0.02, seed=7).toPandas()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sample_all["num_votes"].clip(lower=1), sample_all["average_rating"], s=6, alpha=0.15, label="sample")
ax.scatter(anom_pd["num_votes"].clip(lower=1), anom_pd["average_rating"], s=10, alpha=0.5, label="anomalies")
ax.set_xscale("log")
ax.set_xlabel("num_votes (log)")
ax.set_ylabel("average_rating")
ax.set_title(f"High rating (≥8) with bottom-decile votes (n={anom_n})")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
p4 = charts_dir / "rating_vote_anomalies.png"
save_and_show(fig, p4)
anom.limit(10).toPandas()

## 5. Pre-release signal lift and hit-label sensitivity

The lift chart uses director experience available before each film. It replaces the circular vote-quartile comparison because vote count is part of the hit outcome.

In [ ]:
labeled = with_hit_label(movies)
hit_rate = labeled.agg(F.avg("is_hit")).first()[0]
signal_lift_rows = [
    row.asDict(recursive=True)
    for row in pre_release_signal_lift(
        labeled,
        signal_col="director_prior_movie_count_mean",
        buckets=4,
    ).collect()
]
signal_lift = pd.DataFrame(signal_lift_rows)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    signal_lift["signal_band"].astype(str),
    signal_lift["lift_vs_eligible"],
    color="C2",
)
ax.axhline(1.0, color="black", linewidth=1, linestyle="--", label="eligible baseline")
ax.set_xlabel("Director prior-film-count quartile (1 = least experience)")
ax.set_ylabel("Hit-rate lift")
ax.set_title("Hit-rate lift by pre-release director experience")
for bar, film_count in zip(bars, signal_lift["film_count"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"n={film_count:,}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
p5 = charts_dir / "pre_release_signal_lift.png"
save_and_show(fig, p5)

sensitivity_expressions = []
for rating_min in [6.5, 7.0, 7.5]:
    for votes_min in [500, 1000, 2000]:
        alias = f"rating_{rating_min}_votes_{votes_min}"
        sensitivity_expressions.append(
            F.avg(
                (
                    (F.col("average_rating") >= F.lit(rating_min))
                    & (F.col("num_votes") >= F.lit(votes_min))
                ).cast("int")
            ).alias(alias)
        )
sensitivity_values = movies.agg(*sensitivity_expressions).collect()[0].asDict()
hit_label_sensitivity = [
    {
        "rating_min": rating_min,
        "votes_min": votes_min,
        "positive_rate": float(
            sensitivity_values[f"rating_{rating_min}_votes_{votes_min}"]
        ),
    }
    for rating_min in [6.5, 7.0, 7.5]
    for votes_min in [500, 1000, 2000]
]
sensitivity = pd.DataFrame(hit_label_sensitivity)
sensitivity_matrix = sensitivity.pivot(
    index="rating_min",
    columns="votes_min",
    values="positive_rate",
)
fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(sensitivity_matrix.values, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(sensitivity_matrix.columns)), sensitivity_matrix.columns)
ax.set_yticks(range(len(sensitivity_matrix.index)), sensitivity_matrix.index)
ax.set_xlabel("Minimum IMDb votes")
ax.set_ylabel("Minimum IMDb rating")
ax.set_title("Audience-reception hit prevalence sensitivity")
for row_index in range(len(sensitivity_matrix.index)):
    for column_index in range(len(sensitivity_matrix.columns)):
        ax.text(
            column_index,
            row_index,
            f"{sensitivity_matrix.iloc[row_index, column_index]:.1%}",
            ha="center",
            va="center",
        )
fig.colorbar(image, ax=ax, label="Positive rate")
fig.tight_layout()
p6 = charts_dir / "hit_label_sensitivity.png"
save_and_show(fig, p6)

print("base hit rate", hit_rate)
display(signal_lift)
display(sensitivity)

In [ ]:
genre_summary = []
for genre in top_genres:
    history = gdec_pd[gdec_pd["genre"] == genre].sort_values("decade")
    if history.empty:
        continue
    first = history.iloc[0]
    latest = history.iloc[-1]
    genre_summary.append(
        {
            "genre": genre,
            "first_decade": int(first["decade"]),
            "first_median_rating": float(first["median_rating"]),
            "latest_decade": int(latest["decade"]),
            "latest_median_rating": float(latest["median_rating"]),
            "rating_change": float(
                latest["median_rating"] - first["median_rating"]
            ),
            "total_films": int(history["film_count"].sum()),
        }
    )

runtime_peak = rt.sort_values("film_count", ascending=False).iloc[0]
metrics = {
    "schema_version": 2,
    "job": "core_analytics",
    "input": str(paths.movies_awards_enriched_dir),
    "row_count": n,
    "genre_summary": genre_summary,
    "director_prior_rating_corr": corr,
    "anomaly_count_rating_ge_8_bottom_decile_votes": anom_n,
    "base_hit_rate": float(hit_rate) if hit_rate is not None else None,
    "runtime_profile": {
        "peak_volume_bucket_minutes": int(runtime_peak["runtime_bucket"]),
        "peak_volume_film_count": int(runtime_peak["film_count"]),
        "peak_volume_median_rating": float(runtime_peak["median_rating"]),
        "minimum_films_per_bucket": 50,
    },
    "pre_release_signal_lift": signal_lift_rows,
    "hit_label_sensitivity": hit_label_sensitivity,
    "charts": [str(path) for path in [p1, p2, p3, p4, p5, p6]],
    "notes": {
        "environment": "local Spark notebook, not Dataproc JupyterHub",
        "hit_label": "average_rating >= 7.0 AND num_votes >= 1000",
        "director_correlation": "descriptive association, not causation",
        "runtime": "descriptive profile, not a universal causal sweet spot",
        "anomalies": "niche or high-variance candidates, not evidence of manipulation",
        "signal_lift": "uses director prior film count; no post-release vote bands",
    },
}

display(Markdown("### Core analytics: corrected results summary"))
display(pd.DataFrame(
    [
        ("Movies analyzed", metrics["row_count"]),
        ("Director prior-rating correlation", round(corr, 4)),
        ("High-rating / low-vote niche candidates", anom_n),
        ("Base hit rate", None if hit_rate is None else round(hit_rate, 4)),
        ("Peak-volume runtime bucket", metrics["runtime_profile"]["peak_volume_bucket_minutes"]),
    ],
    columns=["Result", "Value"],
))

display(Markdown("### Interpretation notes"))
display(pd.DataFrame(list(metrics["notes"].items()), columns=["Note", "Value"]))

display(Markdown("### Report-ready charts"))
display(pd.DataFrame(
    {
        "Chart": [
            "Genre and decade median rating",
            "Director prior rating versus film rating",
            "Runtime profile",
            "High-rating / low-vote niche candidates",
            "Pre-release director-experience lift",
            "Hit-label sensitivity",
        ],
        "File": [p1.name, p2.name, p3.name, p4.name, p5.name, p6.name],
    }
))

In [ ]:
out = metrics_dir / "analytics_metrics.json"
out.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print("Saved corrected analytics metrics:", out)
spark.stop()